In [1]:
import pathlib

import numpy as np
import pandas as pd
import pyarrow as pa
import swifter  # noqa: F401

from pyarrow import parquet as pq
from rdkit import Chem
from tqdm.auto import tqdm

In [ ]:
output_file = "quantumpioneer_transition_state_semiempirical_results.parquet"

In [3]:
cwd = pathlib.Path().absolute()
quantum_green = pathlib.Path("/home/shared/projects/quantum_green")
paquets = quantum_green / "supercloud_xfer"
data_dir = cwd.parent.parent / "data" / "qm_results"

In [4]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


def swifter_apply(seris, func, desc="Applying function"):
    return seris.swifter.progress_bar(True, desc=desc).apply(func)


def extract_last(lst):
    if isinstance(lst, list) and len(lst) > 0:
        return lst[-1]
    return lst

In [5]:
ts_data = pd.read_pickle(quantum_green / "ts" / "quantum_green_ts_data_24july2c.pkl")
head(ts_data)

,ts_dft_log_source_utf_8_sha512,dft_table_id,ts_dft_log_path,ts_dft_route_section,charge,multiplicity,ts_dft_opt_max_steps,ts_dft_opt_normal_termination_check,ts_dft_opt_cpu_time,ts_dft_opt_wall_time,ts_dft_sum_of_electronic_and_thermal_enthalpies_hartree,ts_dft_hartreefock_energy_hartree,ts_dft_zpe_hartree,ts_dft_e0_zpe_hartree,ts_dft_gibbs_hartree,ts_dft_scf_hartree,ts_dft_frequency_modes,ts_dft_std_forces,ts_dft_opted_std_xyz,ts_dft_opted_xyz_input_orientation,is_ts,batch_label,ts_dft_log_filename_id,rsmi,psmi,r1smi,r2smi,p1smi,p2smi,r1nasmi,r2nasmi,p1nasmi,p2nasmi,std_xyz_str,xyz_str,p1_reaction_center_atom_map_num,r1_reaction_center_atom_map_num,h_atom_map_num,r1_matched_sha512,r2_matched_sha512,p1_matched_sha512,p2_matched_sha512,r1_matched_smi,r1_matched_std_xyz_str,r2_matched_smi,r2_matched_std_xyz_str,p1_matched_smi,p1_matched_std_xyz_str,p2_matched_smi,p2_matched_std_xyz_str,connectivity_check_r1,connectivity_check_r2,connectivity_check_r1_dist,connectivity_check_r2_dist,r2_h_idx,r2_reacting_site_idx,r2_reacting_site_element,r2_reacting_site_h_bond_distance,r2_ts_reacting_site_h_bond_distance,r2_ts_reacting_site_h_bond_distance_pct_change,connectivity_check_p1,connectivity_check_p2,connectivity_check_p1_dist,connectivity_check_p2_dist,p2_h_idx,p2_reacting_site_idx,p2_reacting_site_element,p2_reacting_site_h_bond_distance,p2_ts_reacting_site_h_bond_distance,p2_ts_reacting_site_h_bond_distance_pct_change,rxn_smi,fingerprint,rxn_smi_original_order,rdmc_large_factor,rdmc_small_factor,neg_freq,neg_freq_z_score,rp_connectivity_check_true_sum,connectivity_check_r1_score,connectivity_check_r2_score,connectivity_check_p2_score,connectivity_check_p1_score,ho2_ts_h_mode,ho2_non_react_h_mode,ho2_ts_h_score,ho2_non_react_h_score,ho2_non_div_ts_mode_ratio,r1_dft_e0_zpe_hartree,r2_dft_e0_zpe_hartree,p1_dft_e0_zpe_hartree,p2_dft_e0_zpe_hartree,rcomplex_dft_e0_zpe_hartree,pcomplex_dft_e0_zpe_hartree,dft_fwd_barrier_e0_zpe_hartree,dft_rev_barrier_e0_zpe_hartree,dft_fwd_Hrxn_e0_zpe_hartree,dft_fwd_barrier_e0_zpe_kcal,dft_rev_barrier_e0_zpe_kcal,dft_fwd_Hrxn_e0_zpe_kcal,ts_dlpno_log_filename_id,ts_dlpno_xyz_tuple,ts_dlpno_log_path,ts_dlpno_sp_hartree,dft_opted_xyz_tuple_used_for_dlpno_calc,r1_dlpno_sp_hartree,r2_dlpno_sp_hartree,p1_dlpno_sp_hartree,p2_dlpno_sp_hartree,all_species_has_dlpno_sp,fwd_Hrxn_dlpno_sp_hartree,ts_and_species_has_dlpno_sp,fwd_barrier_dlpno_sp_hartree,rev_barrier_dlpno_sp_hartree,fwd_Hrxn_dlpno_sp_kcal,fwd_barrier_dlpno_sp_kcal,rev_barrier_dlpno_sp_kcal,r1_dft_log_path,r2_dft_log_path,p1_dft_log_path,p2_dft_log_path,r1_dlpno_log_path,r2_dlpno_log_path,p1_dlpno_log_path,p2_dlpno_log_path,r1_dft_hartreefock_energy_hartree,r2_dft_hartreefock_energy_hartree,p1_dft_hartreefock_energy_hartree,p2_dft_hartreefock_energy_hartree,r1_dft_zpe_hartree,r2_dft_zpe_hartree,p1_dft_zpe_hartree,p2_dft_zpe_hartree,ts_and_species_has_dft_zpe,species_has_dft_zpe,fwd_Hrxn_dft_zpe_hartree,fwd_barrier_dft_zpe_hartree,rev_barrier_dft_zpe_hartree,fwd_Hrxn_dlpno_sp_dft_zpe_hartree,fwd_barrier_dlpno_sp_dft_zpe_hartree,rev_barrier_dlpno_sp_dft_zpe_hartree,fwd_Hrxn_dlpno_sp_dft_zpe_kcal,fwd_barrier_dlpno_sp_dft_zpe_kcal,rev_barrier_dlpno_sp_dft_zpe_kcal,r1_dft_zpe_scaled_hartree,r2_dft_zpe_scaled_hartree,p1_dft_zpe_scaled_hartree,p2_dft_zpe_scaled_hartree,ts_dft_zpe_scaled_hartree,fwd_barrier_dlpno_sp_dft_zpe_scaled_hartree,rev_barrier_dlpno_sp_dft_zpe_scaled_hartree,fwd_Hrxn_dlpno_sp_dft_zpe_scaled_hartree,rev_Hrxn_dlpno_sp_dft_zpe_scaled_hartree,fwd_barrier_dlpno_sp_dft_zpe_scaled_kcal,rev_barrier_dlpno_sp_dft_zpe_scaled_kcal,fwd_Hrxn_dlpno_sp_dft_zpe_scaled_kcal,rev_Hrxn_dlpno_sp_dft_zpe_scaled_kcal,ts_dft_frequencies,r1_dft_frequencies,r2_dft_frequencies,p1_dft_frequencies,p2_dft_frequencies,smiles
0,4c648a435eb66f655a9afa7aebe1b6288cfbe5d4377ab7...,545983,/home/gridsan/groups/RMG/Projects/Habs/data/ts...,"P opt=(ts,calcall,maxcycle=64,noeig,nomicro,ca...",0,2,64,True,53637.5,1178.8,-499.312919,-499.513846,0.187656,-499.326190,-499.365824,4

Contains 167237 rows


In [6]:
matching_df = pd.read_csv(cwd / "quantum_green_ts_data_match.csv")
matching_df["semiempirical_index"] = np.where(
    matching_df["xtb_index"] >= 0,
    matching_df["xtb_index"],
    np.where(
        matching_df["pm7_index"] >= 0,
        matching_df["pm7_index"],
        matching_df["am1_index"],
    ),
)
matching_df = (
    matching_df[matching_df["semiempirical_index"] >= 0]
    .sort_values("semiempirical_index")
    .reset_index(drop=True)
    .copy()
)
matching_df["smiles"] = matching_df["dft_table_id"].map(
    ts_data.set_index("dft_table_id")["smiles"]
)
head(matching_df)

,dft_table_id,dft_index,dlpno_index,am1_index,pm7_index,xtb_index,semiempirical_index,smiles
0,494252,434,99708,11,12,13,13,[O:1]([C:4]([O:2][O:3])([H:5])[H:6])[H:7].[C:8...
1,494257,439,79845,29,30,31,31,[O:1]([C:4]([O:2][O:3])([H:5])[H:6])[H:7].[C:8...


Contains 167200 rows


In [7]:
def read_column(name: str):
    table = pa.concat_tables(
        [
            pq.read_table(
                paquets / f"semiempirical_ts_semiempirical_{case}.parquet",
                columns=[name],
            )
            for case in ["v9", "nho_oho_1_v9", "nho_oho_2_v9", "nho_oho_3_v9"]
        ]
    )[name]
    return table

In [8]:
normal_termination = read_column("normal_termination")
keep_index = np.zeros(len(normal_termination), dtype=bool)
keep_index[matching_df["semiempirical_index"].values] = True
keep_index = np.logical_and(keep_index, normal_termination.to_pylist())
row_filter = pa.array(keep_index)
sum(keep_index).item(), row_filter.sum().as_py()

(167200, 167200)

In [9]:
column_metadata = {
    "smiles": "Reaction SMILES (with atom mapping indicating xyz and force order)",
    "route_section": "Level of theory",
    "charge": "Molecular formal charge",
    "multiplicity": "Electron multiplicity",
    "max_steps": "Maximum allowed steps",
    "cpu_time": "CPU time in seconds",
    "wall_time": "Wall time in seconds",
    "e0_h": "Enthalpy at 298K",
    "hf": "E0 for non-wavefunction methods",
    "zpe_per_atom": "Per-atom zero point energy",
    "e0_zpe": "Gibbs free energy at 0K",
    "gibbs": "Gibbs free energy at 298K",
    "frequencies": "Vibrational frequencies",
    "frequency_modes": "Vibrational modes",
    "initial_xyz": "Input XYZ coords",
    "std_xyz": "Standardized XYZ coords after optimization",
    "std_forces": "Standardized forces after optimization",
}

In [10]:
semiempirical_data = pa.Table.from_arrays(
    [pa.array(matching_df["smiles"])],
    schema=pa.schema([pa.field("smiles", pa.string())], metadata=column_metadata),
)

pbar = tqdm([key for key in column_metadata.keys() if key != "smiles"])
for name in pbar:
    pbar.set_description(f"Processing column {name}")
    column = read_column(name).filter(row_filter)
    if name in ["std_xyz", "std_forces"]:
        column = pa.array([extract_last(item.as_py()) for item in column])
    old_semiempirical_data = semiempirical_data
    semiempirical_data = old_semiempirical_data.append_column(name, column)
    del old_semiempirical_data
    del column

  0%|          | 0/16 [00:00<?, ?it/s]

In [11]:
semiempirical_data.schema

smiles: string
route_section: string
charge: uint8
multiplicity: uint8
max_steps: uint32
cpu_time: uint32
wall_time: uint32
e0_h: double
hf: double
zpe_per_atom: double
e0_zpe: double
gibbs: double
frequencies: list<element: double>
  child 0, element: double
frequency_modes: list<element: list<element: list<element: double>>>
  child 0, element: list<element: list<element: double>>
      child 0, element: list<element: double>
          child 0, element: double
initial_xyz: list<element: list<element: float>>
  child 0, element: list<element: float>
      child 0, element: float
std_xyz: list<item: list<item: double>>
  child 0, item: list<item: double>
      child 0, item: double
std_forces: list<item: list<item: double>>
  child 0, item: list<item: double>
      child 0, item: double
-- schema metadata --
smiles: 'Reaction SMILES (with atom mapping indicating xyz and force orde' + 2
route_section: 'Level of theory'
charge: 'Molecular formal charge'
multiplicity: 'Electron multiplici

In [12]:
semiempirical_df = semiempirical_data.to_pandas()
head(semiempirical_df)

,smiles,route_section,charge,multiplicity,max_steps,cpu_time,wall_time,e0_h,hf,zpe_per_atom,e0_zpe,gibbs,frequencies,frequency_modes,initial_xyz,std_xyz,std_forces
0,[O:1]([C:4]([O:2][O:3])([H:5])[H:6])[H:7].[C:8...,"P opt=(ts,calcall,maxcycle=90,noeig,nomicro,ca...",0.0,2.0,90.0,57,9,-42.513596,-42.696474,0.169243,-42.527231,-42.568551,"[-781.8643, 32.471, 50.5414, 60.661, 91.1673, ...","[[[1.0, 8.0, -0.0, 0.0, -0.0], [2.0, 8.0, 0.02...","[[1.0, 8.0, 0.0, -1.794081, -1.656718, -0.0569...","[[1.0, 8.0, 0.0, -2.1198270320892334, -0.47277...","[[1.0, 8.0, 3.769999921132694e-07, 1.641000039..."
1,[O:1]([C:4]([O:2][O:3])([H:5])[H:6])[H:7].[C:8...,"P opt=(ts,calcall,maxcycle=90,noeig,nomicro,ca...",0.0,2.0,90.0,153,23,-43.551291,-43.750695,0.182283,-43.568411,-43.617631,"[-988.7339, 17.3197, 19.392, 30.6842, 41.6583,...","[[[1.0, 8.0, 0.0, -0.0, 0.0], [2.0, 8.0, 0.01,...","[[1.0, 8.0, 0.0, 0.000181, -0.076914, -0.82913...","[[1.0, 8.0, 0.0, -4.520959854125977, -1.711431...","[[1.0, 8.0, 6.419999749596172e-07, -3.69100007..."


Contains 167200 rows


In [13]:
rows_to_delete = set()
for col in semiempirical_df.columns:
    to_delete = semiempirical_df.index[semiempirical_df[col].isna()]
    rows_to_delete.update(to_delete)
    if len(to_delete) > 0:
        print(f"Column {col} has {len(to_delete)} missing values")

print(f"\nMarked {len(rows_to_delete)} rows for deletion")

Column charge has 1 missing values
Column multiplicity has 1 missing values
Column max_steps has 2 missing values
Column std_xyz has 2 missing values

Marked 4 rows for deletion


In [14]:
energy_cols = ["e0_h", "hf", "e0_zpe", "gibbs"]
mean_energy = semiempirical_df[energy_cols].mean(axis=1)
for col in energy_cols:
    deviations = (semiempirical_df[col] - mean_energy).abs()
    rows_to_delete.update(deviations.index[deviations > 1.0])

print(f"\nA total of {len(rows_to_delete)} rows are now marked for deletion.")


A total of 5 rows are now marked for deletion.


In [15]:
PARAMS = Chem.SmilesParserParams()
PARAMS.removeHs = False


def get_elements_from_smiles(smiles):
    reactant_smiles = smiles.split(">>")[0].split(".")
    mols = [Chem.MolFromSmiles(smi, PARAMS) for smi in reactant_smiles]
    order = {
        atom.GetAtomMapNum(): atom.GetSymbol()
        for mol in mols
        for atom in mol.GetAtoms()
    }
    try:
        return "".join([order[i] for i in range(1, len(order) + 1)])
    except TypeError:
        return ""


PERIODIC_TABLE = Chem.GetPeriodicTable()


def get_elements_from_xyz(xyz):
    if xyz is None:
        return None
    return "".join(PERIODIC_TABLE.GetElementSymbol(int(item[1])) for item in xyz)

In [16]:
elements_from_xyz = swifter_apply(
    semiempirical_df["std_xyz"], get_elements_from_xyz, "Getting elements from xyz"
)
elements_from_smiles = swifter_apply(
    semiempirical_df["smiles"],
    get_elements_from_smiles,
    "Getting elements from smiles",
)
rows_to_delete |= set(semiempirical_df.index[elements_from_smiles != elements_from_xyz])

print(f"\nA total of {len(rows_to_delete)} rows are now marked for deletion.")

Getting elements from xyz:   0%|          | 0/167200 [00:00<?, ?it/s]

Getting elements from smiles:   0%|          | 0/167200 [00:00<?, ?it/s]


A total of 37 rows are now marked for deletion.


In [17]:
keep_index = np.ones(len(semiempirical_df), dtype=bool)
keep_index[np.array(list(rows_to_delete), dtype=int)] = False
filtered_semiempirical_data = semiempirical_data.filter(pa.array(keep_index))
del semiempirical_data

In [18]:
data_dir.mkdir(parents=True, exist_ok=True)
pq.write_table(filtered_semiempirical_data, data_dir / output_file, compression="zstd")

In [19]:
parquet_file = pq.ParquetFile(data_dir / output_file)
print(f"Successfully wrote parquet file with {parquet_file.metadata.num_rows} rows")

Successfully wrote parquet file with 167163 rows
